# 🧠 LM World Model — Full Research Pipeline
**`Qwen2.5-1.5B` | Unsloth SFT | RL Simulation | MechInterp`**

> *"Do Language Models Construct Internal World Models?"*  
> — Tinevimbo Musingadi, HIT Research Project (2026)

This notebook runs the **complete research pipeline** end-to-end on Google Colab:

| Stage | What happens |
|---|---|
| **0. Setup** | Clone repo, install deps, mount Drive |
| **1. Data** | Load & inspect the 10k Phase 1 trace dataset |
| **2. SFT** | Fine-tune Qwen2.5-1.5B via Unsloth (4× faster than HF) |
| **3. Evaluation** | Output Accuracy, Trace Accuracy, Levenshtein, Temp Test |
| **4. RL Simulation** | GRPO-style reward signal on trace correctness |
| **5. MechInterp** | Logit Lens, Linear Probing, Activation Patching |

**Runtime:** T4 GPU (free tier) — ~4-6 hrs for full SFT

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 0A — Clone Repo & Mount Google Drive
# ════════════════════════════════════════════════════════════
import os

# Clone the repository (skip if already present)
if not os.path.exists('/content/lm-world-model'):
    !git clone https://github.com/TinevimboMusingadi/lm-world-model.git /content/lm-world-model

# Mount Google Drive for saving checkpoints across sessions
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = '/content/drive/MyDrive/lm-world-model'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/interp', exist_ok=True)

# Work from the repo root so all relative paths (data/, training/, eval/) just work
%cd /content/lm-world-model

print(f'✅ Repo ready at {os.getcwd()}')
print(f'✅ Drive mounted — checkpoints → {DRIVE_DIR}/checkpoints')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 0B — Install Dependencies
# Unsloth gives ~4× SFT speedup over vanilla HF on T4/A100
# ════════════════════════════════════════════════════════════
!pip install -q unsloth
!pip install -q --upgrade transformers datasets trl peft bitsandbytes
!pip install -q wandb jsonlines pydantic nnsight nnterp plotly Levenshtein

import torch
import transformers
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Transformers {transformers.__version__}')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 1A — Load & Inspect the Phase 1 Trace Dataset
# ════════════════════════════════════════════════════════════
import json
import jsonlines
import pandas as pd
import plotly.express as px
from collections import Counter

DATA_FILE = 'data/splits/phase1_raw.jsonl'

records = []
with jsonlines.open(DATA_FILE) as reader:
    for r in reader:
        records.append(r)

df = pd.DataFrame([{
    'program_id': r['program_id'],
    'split': r['split'],
    'complexity': r['complexity'],
    'trace_steps': len([l for l in r['execution_trace'].split('\n') if l.strip().startswith('[line=') and '=' in l[7:]]),
    'code_lines': len(r['code'].strip().split('\n')),
    'has_branch': 'if ' in r['code'],
    'output': r['output'],
} for r in records])

print(f'📊 Total records: {len(df):,}')
print(df['split'].value_counts().to_string())

# Plot 1: Trace step distribution by complexity
fig1 = px.histogram(df, x='trace_steps', color='complexity', barmode='overlay',
                    title='Execution Trace Length Distribution (Steps) by Complexity',
                    labels={'trace_steps': 'Execution Steps', 'complexity': 'Complexity Level'},
                    nbins=40, opacity=0.75)
fig1.show()

# Plot 2: Split breakdown
fig2 = px.bar(df['split'].value_counts().reset_index(),
              x='split', y='count',
              title='Dataset Split Distribution',
              color='split')
fig2.show()

# Quick sample
print('\n📋 Sample program (complexity=2, with branch):')
sample = df[(df['complexity']==2) & (df['has_branch'])].iloc[0]
r = records[df.index[(df['program_id']==sample['program_id'])].tolist()[0]]
print(f"CODE:\n{r['code']}")
print(f"\nTRACE:\n{r['execution_trace']}")
print(f"\nOUTPUT: {r['output']}")

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 1B — Build SFT Training Format
# Shows exactly what the model will see during training
# ════════════════════════════════════════════════════════════
SYSTEM_PROMPT = (
    "You are a neural computer that executes Python code. "
    "Given a program, produce the full execution trace showing the state of all variables "
    "after each line, then give the final output inside <o> tags."
)

def build_sft_example(record, condition='B'):
    """
    Condition A: model sees code, must predict output only.
    Condition B: model sees code, must predict full trace + output. (MAIN EXPERIMENT)
    Condition C: model sees code + partial masked trace, must complete it.
    """
    code_block = f"<code>\n{record['code']}\n</code>"
    trace_block = f"<execution_trace>\n{record['execution_trace']}\n</execution_trace>"
    output_block = f"<o>{record['output']}</o>"

    if condition == 'A':
        return f"{code_block}\n\n{output_block}"
    elif condition == 'B':
        return f"{code_block}\n\n{trace_block}\n\n{output_block}"

def format_for_unsloth(record, condition='B'):
    return {
        "instruction": SYSTEM_PROMPT,
        "input": f"<code>\n{record['code']}\n</code>",
        "output": build_sft_example(record, condition)
    }

sample_fmt = format_for_unsloth(records[5], 'B')
print('═══ INSTRUCTION ═══')
print(sample_fmt['instruction'])
print('\n═══ INPUT ═══')
print(sample_fmt['input'])
print('\n═══ TARGET OUTPUT ═══')
print(sample_fmt['output'])

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 2A — Load Qwen2.5-1.5B via Unsloth (4× faster SFT)
# Unsloth handles QLoRA, gradient checkpointing, & RoPE scaling
# ════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel

MODEL_NAME   = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN  = 1024
LOAD_IN_4BIT = True   # Keeps VRAM to ~4-6GB on T4
LORA_R       = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # Auto-detect: bf16 on Ampere, fp16 on T4
)

# Apply LoRA with Unsloth's optimised PEFT
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Unsloth's version uses 30% less VRAM
    random_state=42,
)
model.print_trainable_parameters()
print(f'✅ Model loaded: {MODEL_NAME} in 4-bit QLoRA')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 2B — Prepare Dataset & Run SFT Training
# Condition B: Full trace supervision (the key experiment)
# ════════════════════════════════════════════════════════════
import wandb
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# W&B logging
wandb.init(project='lm-world-model', name='phase1-qwen1.5b-condB-unsloth', config={
    'model': MODEL_NAME, 'condition': 'B', 'lora_r': LORA_R, 'max_seq_len': MAX_SEQ_LEN
})

CONDITION  = 'B'
CHECKPOINT = f'{DRIVE_DIR}/checkpoints/phase1_qwen1.5b_condB'

# Build train split
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS = tokenizer.eos_token

def format_record(r):
    ex = format_for_unsloth(r, CONDITION)
    return {'text': alpaca_prompt.format(**ex) + EOS}

train_records = [r for r in records if r['split'] == 'train']
hf_dataset = Dataset.from_list([format_record(r) for r in train_records])

print(f'Training samples: {len(hf_dataset):,}')
print(f'Sample text (first 400 chars):')
print(hf_dataset[0]['text'][:400])

# ─── Train ───
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    args=SFTConfig(
        output_dir=CHECKPOINT,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # Effective batch=16
        warmup_steps=50,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=3,
        report_to='wandb',
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='text',
        optim='adamw_8bit',    # Unsloth 8-bit Adam
        weight_decay=0.01,
        lr_scheduler_type='cosine',
    ),
)

trainer_stats = trainer.train()
print(f'\n✅ Training complete!')
print(f'   Runtime: {trainer_stats.metrics["train_runtime"]:.2f}s')
print(f'   Samples/s: {trainer_stats.metrics["train_samples_per_second"]:.2f}')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 2C — Plot Training Loss Curve (Grokking Detection)
# Look for the characteristic 'late generalization' phase transition
# ════════════════════════════════════════════════════════════
import plotly.graph_objects as go

log_history = trainer.state.log_history
steps = [x['step'] for x in log_history if 'loss' in x]
losses = [x['loss'] for x in log_history if 'loss' in x]

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines+markers',
                         name='Training Loss', line=dict(color='#EF553B', width=2)))
fig.update_layout(
    title='Training Loss Curve — Phase 1 SFT (Condition B)',
    xaxis_title='Step', yaxis_title='Cross-Entropy Loss',
    template='plotly_dark',
    annotations=[dict(x=max(steps)//2, y=min(losses)+0.1,
                      text='Watch for Grokking: sudden drop in loss', showarrow=False,
                      font=dict(color='white'))]
)
fig.show()

# Save to Drive
import json
with open(f'{DRIVE_DIR}/results/training_log.json', 'w') as f:
    json.dump(log_history, f)
print(f'📁 Training logs saved to Drive')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 3A — Inference & Evaluation
# Tests: Output Accuracy, Trace Accuracy, Levenshtein Distance
# ════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel
import Levenshtein
import re

FastLanguageModel.for_inference(model)  # Enable Unsloth's 2× inference optimisation

def extract_output(text):
    m = re.search(r'<o>(.*?)</o>', text, re.DOTALL)
    return m.group(1).strip() if m else ''

def extract_trace(text):
    m = re.search(r'<execution_trace>(.*?)</execution_trace>', text, re.DOTALL)
    return m.group(1).strip() if m else ''

def evaluate_records(eval_records, temperature=0.0, n_samples=100, label='test_indist'):
    results = []
    for r in eval_records[:n_samples]:
        prompt = alpaca_prompt.format(
            instruction=SYSTEM_PROMPT,
            input=f"<code>\n{r['code']}\n</code>",
            output='')  # We leave output blank — model must generate it

        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        gen_kwargs = dict(max_new_tokens=512, pad_token_id=tokenizer.eos_token_id,
                          do_sample=(temperature > 0.0))
        if temperature > 0.0:
            gen_kwargs['temperature'] = temperature
        else:
            gen_kwargs['temperature'] = None  # Greedy

        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)

        generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        pred_output = extract_output(generated)
        pred_trace  = extract_trace(generated)
        true_output = r['output']
        true_trace  = r['execution_trace']

        output_correct = (pred_output.strip() == true_output.strip())
        lev_dist = Levenshtein.distance(pred_trace, true_trace)

        results.append({
            'program_id': r['program_id'],
            'complexity': r['complexity'],
            'output_correct': output_correct,
            'levenshtein': lev_dist,
            'temperature': temperature,
            'split': label,
        })

    oa = sum(r['output_correct'] for r in results) / len(results) * 100
    avg_lev = sum(r['levenshtein'] for r in results) / len(results)
    print(f'[T={temperature:.1f} | {label}] OA={oa:.1f}% | Avg LevenshteinDist={avg_lev:.1f}')
    return results

# Run evaluations at T=0 and T=0.5 (Temperature Sensitivity Test)
test_indist = [r for r in records if r['split'] == 'test_indist']
test_ood    = [r for r in records if r['split'] == 'test_ood']

res_t0   = evaluate_records(test_indist, temperature=0.0, n_samples=100, label='test_indist')
res_t05  = evaluate_records(test_indist, temperature=0.5, n_samples=100, label='test_indist')
res_ood  = evaluate_records(test_ood,    temperature=0.0, n_samples=100, label='test_ood')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 3B — Evaluation Visualizations
# ════════════════════════════════════════════════════════════
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

all_results = res_t0 + res_t05 + res_ood
df_res = pd.DataFrame(all_results)
df_res.to_csv(f'{DRIVE_DIR}/results/eval_results.csv', index=False)

# Summary table
summary = df_res.groupby(['split', 'temperature']).agg(
    output_accuracy=('output_correct', lambda x: x.mean()*100),
    avg_levenshtein=('levenshtein', 'mean')
).round(2)
print('\n=== EVALUATION SUMMARY ===')
print(summary.to_string())

# Plot: Accuracy vs Temperature (Rigour Test from thing_to_out_for.md)
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Output Accuracy by Temperature', 'Levenshtein Distance (State Fidelity)'))

for split_name, color in [('test_indist', '#00CC96'), ('test_ood', '#EF553B')]:
    sub = df_res[df_res['split'] == split_name]
    agg = sub.groupby('temperature')['output_correct'].mean() * 100
    fig.add_trace(go.Bar(name=split_name, x=[str(t) for t in agg.index],
                         y=agg.values, marker_color=color), row=1, col=1)

lev_agg = df_res.groupby('split')['levenshtein'].mean().reset_index()
fig.add_trace(go.Bar(x=lev_agg['split'], y=lev_agg['levenshtein'],
                     marker_color=['#00CC96', '#EF553B', '#AB63FA'][:len(lev_agg)]), row=1, col=2)

fig.update_layout(title='Phase 1 Evaluation: World Model Fidelity', template='plotly_dark', barmode='group')
fig.show()

# Per-complexity accuracy
acc_by_complexity = df_res.groupby('complexity')['output_correct'].mean() * 100
px.bar(acc_by_complexity.reset_index(), x='complexity', y='output_correct',
       title='Output Accuracy by Complexity Level',
       labels={'output_correct': 'Accuracy (%)', 'complexity': 'Complexity'},
       color='complexity').show()

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 4 — RL Simulation (GRPO-style Reward)
# We simulate an RL loop: the reward = trace correctness score
# This is a research prototype demonstrating the training signal;
# full GRPO requires multiple GPU workers (production setup).
# ════════════════════════════════════════════════════════════
import Levenshtein
import random

def compute_trace_reward(generated_text, ground_truth_record):
    """
    Reward function for RL training.
    Score = normalized correctness of generated trace vs ground truth.
    R=1.0: perfect trace, R=0.0: completely wrong trace.
    """
    pred_output = extract_output(generated_text)
    pred_trace  = extract_trace(generated_text)
    true_output = ground_truth_record['output']
    true_trace  = ground_truth_record['execution_trace']

    # Output accuracy bonus
    output_bonus = 1.0 if pred_output.strip() == true_output.strip() else 0.0

    # Trace fidelity via normalized Levenshtein (1 = identical, 0 = entirely different)
    max_len = max(len(pred_trace), len(true_trace), 1)
    lev_score = 1.0 - (Levenshtein.distance(pred_trace, true_trace) / max_len)

    # Combined reward: weighted sum
    reward = 0.4 * output_bonus + 0.6 * max(0.0, lev_score)
    return reward

# Simulate reward distribution over a small batch
rl_sample = random.sample(records, 50)
rl_rewards = []

FastLanguageModel.for_inference(model)
for r in rl_sample[:20]:  # Small rollout for demo
    prompt = alpaca_prompt.format(
        instruction=SYSTEM_PROMPT,
        input=f"<code>\n{r['code']}\n</code>",
        output='')
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.8,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reward = compute_trace_reward(generated, r)
    rl_rewards.append({'program_id': r['program_id'], 'complexity': r['complexity'], 'reward': reward})

df_rl = pd.DataFrame(rl_rewards)
print(f'\nRL Reward Statistics:')
print(df_rl['reward'].describe().round(3))

px.histogram(df_rl, x='reward', color='complexity', barmode='overlay',
             title='RL Reward Distribution (GRPO Surrogate)',
             labels={'reward': 'Trace Correctness Reward (0-1)'},
             nbins=20).show()

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 5A — Mechanistic Interpretability: Logit Lens
# Watch the model's token prediction evolve across layers
# ════════════════════════════════════════════════════════════
# Note: nnterp works best on the base (non-PEFT) model.
# We re-load a fresh copy here for clean activation access.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
import plotly.graph_objects as go

# Merge LoRA for cleaner activation analysis
merged_model_path = f'{DRIVE_DIR}/checkpoints/phase1_qwen1.5b_condB_merged'
if not os.path.exists(merged_model_path):
    print('Merging LoRA weights for clean activation analysis...')
    merged = model.merge_and_unload()
    merged.save_pretrained(merged_model_path)
    tokenizer.save_pretrained(merged_model_path)
    print(f'✅ Merged model saved to Drive')

# Manual Logit Lens (without nnsight hook overhead)
from transformers import AutoModelForCausalLM

interp_model = AutoModelForCausalLM.from_pretrained(
    merged_model_path, torch_dtype=torch.float16, device_map='auto')
interp_model.eval()

sample_r = [r for r in records if r['complexity']==2 and r['split']=='test_indist'][0]
PROBE_PROMPT = f"""<code>\n{sample_r['code']}\n</code>\n\n<execution_trace>\n"""

inputs = tokenizer(PROBE_PROMPT, return_tensors='pt').to('cuda')
n_tokens = inputs['input_ids'].shape[1]

# Hook all residual stream outputs
stored_residuals = {}
hooks = []

def make_hook(layer_idx):
    def hook_fn(module, input, output):
        stored_residuals[layer_idx] = output[0].detach().cpu().float()
    return hook_fn

for i, layer in enumerate(interp_model.model.layers):
    hooks.append(layer.register_forward_hook(make_hook(i)))

with torch.no_grad():
    _ = interp_model(**inputs)

for h in hooks: h.remove()

# Apply final norm + unembed to each layer's residual at the last token
W_U = interp_model.lm_head.weight.cpu().float()
final_norm = interp_model.model.norm

top1_tokens, top1_probs = [], []
for layer_idx in range(len(interp_model.model.layers)):
    resid = stored_residuals[layer_idx][0, -1, :]  # Last token position
    normed = final_norm(resid.unsqueeze(0).unsqueeze(0).to('cuda')).squeeze().cpu().float()
    logits = W_U @ normed
    probs = torch.softmax(logits, dim=-1)
    top1 = probs.argmax().item()
    top1_tokens.append(tokenizer.decode([top1]))
    top1_probs.append(probs[top1].item())

# Visualize
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(top1_probs))), y=top1_probs,
    mode='lines+markers', text=top1_tokens,
    hovertemplate='Layer %{x}<br>Token: <b>%{text}</b><br>Prob: %{y:.3f}',
    line=dict(color='#AB63FA', width=2)
))
fig.update_layout(
    title=f'🔭 Logit Lens — What the Model "Thinks" at Each Layer<br><sup>Predicting next token after trace prompt start</sup>',
    xaxis_title='Transformer Layer', yaxis_title='Top-1 Token Probability',
    template='plotly_dark'
)
fig.show()
print(f'\nFinal predicted token at each layer:')
for i, (tok, prob) in enumerate(zip(top1_tokens, top1_probs)):
    print(f'  Layer {i:2d}: {repr(tok):20s} ({prob:.3f})')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 5B — Linear Probing: Does the Model Encode Register State?
# Train a Ridge regression probe per layer to decode variable values
# from the model's residual stream activations.
# ════════════════════════════════════════════════════════════
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import re, numpy as np

def parse_trace_steps(trace_str):
    """Parse a trace into list of dicts mapping variable -> int value"""
    steps = []
    for line in trace_str.strip().split('\n'):
        if not line.startswith('[line='): continue
        vals = {}
        for m in re.finditer(r'(var_\d+)=([\-0-9]+)', line):
            try: vals[m.group(1)] = int(m.group(2))
            except: pass
        if vals: steps.append(vals)
    return steps

# Collect (activation, label) pairs across layers
probe_records = [r for r in records if r['split'] == 'test_indist'][:50]

all_layer_acts = {i: [] for i in range(len(interp_model.model.layers))}
var0_labels = []

for r in probe_records:
    steps = parse_trace_steps(r['execution_trace'])
    if not steps or 'var_0' not in steps[-1]: continue
    label = steps[-1]['var_0']
    
    # Use the prompt up to the final state as context
    prompt = f"<code>\n{r['code']}\n</code>\n\n{r['execution_trace']}"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
    
    stored = {}
    h_list = []
    for i, layer in enumerate(interp_model.model.layers):
        h_list.append(layer.register_forward_hook(make_hook(i)))  # reuse hook fn from above
    with torch.no_grad():
        _ = interp_model(**inputs)
    for h in h_list: h.remove()
    
    for i in range(len(interp_model.model.layers)):
        act = stored_residuals[i][0, -1, :].numpy()  # Last token
        all_layer_acts[i].append(act)
    var0_labels.append(label)

# Train probe per layer
r2_scores = []
for layer_idx in range(len(interp_model.model.layers)):
    X = np.array(all_layer_acts[layer_idx])
    y = np.array(var0_labels, dtype=float)
    if len(X) < 10: r2_scores.append(-1); continue
    X_sc = StandardScaler().fit_transform(X)
    scores = cross_val_score(Ridge(alpha=1.0), X_sc, y, cv=3, scoring='r2')
    r2_scores.append(float(scores.mean()))
    print(f'Layer {layer_idx:2d}: R²={scores.mean():.3f}')

# Plot probe accuracy curve
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(r2_scores))), y=r2_scores,
    mode='lines+markers', fill='tozeroy', fillcolor='rgba(99,110,250,0.2)',
    line=dict(color='#636EFA', width=2),
    hovertemplate='Layer %{x}<br>R²=%{y:.3f}'
))
fig.add_hline(y=0, line_dash='dash', line_color='red',
              annotation_text='R²=0 (no information)')
fig.update_layout(
    title='📊 Linear Probe: var_0 Value Encoded in Residual Stream by Layer',
    xaxis_title='Layer', yaxis_title='R² Score (higher = more info encoded)',
    template='plotly_dark'
)
fig.show()
print(f'\n✅ Best probe R² = {max(r2_scores):.3f} at layer {r2_scores.index(max(r2_scores))}')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 5C — Activation Patching (Causal Tracing)
# Clean vs Corrupted: which layer/token causally determines output?
# ════════════════════════════════════════════════════════════
import torch
import numpy as np
import plotly.graph_objects as go

# Pick a paired example: same code but change first variable value
sample = [r for r in records if r['complexity']==1 and r['split']=='test_indist'][0]

clean_input   = f"<code>\n{sample['code']}\n</code>\n\n<execution_trace>\n"
# Corrupt: replace the first value in execution trace
corrupt_code  = sample['code'].replace(str(int(sample['output'])), str(int(sample['output'])+99), 1)
corrupt_input = f"<code>\n{corrupt_code}\n</code>\n\n<execution_trace>\n"

# Helper: get logit for a specific token
target_tok_id = tokenizer(sample['output'], return_tensors='pt').input_ids[0, -1].item()

def get_output_logit(prompt_str):
    inputs = tokenizer(prompt_str, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = interp_model(**inputs).logits[0, -1, :]
    return logits[target_tok_id].item()

clean_logit   = get_output_logit(clean_input)
corrupt_logit = get_output_logit(corrupt_input)
baseline_diff = clean_logit - corrupt_logit
print(f'Clean logit: {clean_logit:.3f} | Corrupt logit: {corrupt_logit:.3f} | Diff: {baseline_diff:.3f}')

# Cache corrupted activations
corrupt_cache = {}
c_inputs = tokenizer(corrupt_input, return_tensors='pt').to('cuda')
c_hooks = []
for i, layer in enumerate(interp_model.model.layers):
    def make_corrupt_hook(idx):
        def fn(module, inp, out):
            corrupt_cache[idx] = out[0].detach()
        return fn
    c_hooks.append(layer.register_forward_hook(make_corrupt_hook(i)))
with torch.no_grad():
    _ = interp_model(**c_inputs)
for h in c_hooks: h.remove()

# Patch one (layer, pos) at a time into the clean run
n_layers = len(interp_model.model.layers)
cl_inputs = tokenizer(clean_input, return_tensors='pt').to('cuda')
n_tokens  = cl_inputs['input_ids'].shape[1]
patch_tokens = min(n_tokens, min(c_inputs['input_ids'].shape[1], 20))

effect_matrix = np.zeros((n_layers, patch_tokens))
for layer_idx in range(n_layers):
    for pos in range(patch_tokens):
        def make_patch_hook(lidx, position):
            def fn(module, inp, out):
                corrupted_act = corrupt_cache[lidx]
                out_new = list(out)
                out_new[0] = out_new[0].clone()
                out_new[0][:, position, :] = corrupted_act[:, min(position, corrupted_act.shape[1]-1), :]
                return tuple(out_new)
            return fn
        h = interp_model.model.layers[layer_idx].register_forward_hook(make_patch_hook(layer_idx, pos))
        with torch.no_grad():
            patched_logit = interp_model(**cl_inputs).logits[0, -1, target_tok_id].item()
        h.remove()
        effect_matrix[layer_idx, pos] = (clean_logit - patched_logit) / (abs(baseline_diff) + 1e-8)

# Visualize
token_strs = [repr(tokenizer.decode([t])) for t in cl_inputs['input_ids'][0, :patch_tokens].tolist()]
fig = go.Figure(data=go.Heatmap(
    z=effect_matrix,
    x=token_strs,
    y=[f'L{i}' for i in range(n_layers)],
    colorscale='RdBu_r', zmin=-1, zmax=1
))
fig.update_layout(
    title='🗺️ Activation Patching Heatmap — Effect of Corrupting Each Layer×Token',
    xaxis_title='Token Position (corrupted)', yaxis_title='Layer',
    template='plotly_dark', height=600
)
fig.show()
fig.write_html(f'{DRIVE_DIR}/interp/patching_heatmap.html')
print('✅ Heatmap saved to Drive')

In [ ]:
# ════════════════════════════════════════════════════════════
# STAGE 6 — Save Everything & Summary
# ════════════════════════════════════════════════════════════
import json

# Save LoRA checkpoint to Drive
trainer.save_model(f'{DRIVE_DIR}/checkpoints/phase1_qwen1.5b_condB_final')
tokenizer.save_pretrained(f'{DRIVE_DIR}/checkpoints/phase1_qwen1.5b_condB_final')

# Save probing results
probe_summary = {'layer': list(range(len(r2_scores))), 'r2': r2_scores}
pd.DataFrame(probe_summary).to_csv(f'{DRIVE_DIR}/interp/probe_r2_by_layer.csv', index=False)

# Save eval results (already done above)

wandb.finish()

print()
print('╔══════════════════════════════════════════════════════╗')
print('║          LM WORLD MODEL — RUN COMPLETE               ║')
print('╠══════════════════════════════════════════════════════╣')
oa_t0 = sum(r['output_correct'] for r in res_t0) / len(res_t0) * 100
oa_ood = sum(r['output_correct'] for r in res_ood) / len(res_ood) * 100
avg_lev = sum(r['levenshtein'] for r in res_t0) / len(res_t0)
best_probe = max(r2_scores)
best_probe_layer = r2_scores.index(best_probe)
print(f'║  Output Accuracy (InDist, T=0):  {oa_t0:5.1f}%            ║')
print(f'║  Output Accuracy (OOD, T=0):     {oa_ood:5.1f}%            ║')
print(f'║  OOD Generalisation Gap:         {oa_t0-oa_ood:5.1f}%            ║')
print(f'║  Avg Levenshtein (InDist):       {avg_lev:5.1f} tokens       ║')
print(f'║  Best var_0 Probe R²:            {best_probe:5.3f} (Layer {best_probe_layer:2d})  ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Checkpoint: {CHECKPOINT[-40:]:40s} ║')
print(f'║  Results:    {DRIVE_DIR}/results                     ║')
print('╚══════════════════════════════════════════════════════╝')